# Groundedness as an eval metric — measured, not vibes

The standard way to score "is this answer grounded in the context?" is to ask
another LLM — a judge grading an essay it didn't write, with known failure
modes (verbosity bias, self-preference, prompt sensitivity).
[TokenPath](https://tokenpath.ai) gives you a different signal: **measured
attention** — how much attribution mass each claim in the answer actually has
in the source. No judge, deterministic, and it comes with the exact source span
so failures are debuggable.

This notebook turns attribution into scalar metrics you can rank on in an eval
harness, and shows the metric separating grounded, distorted, and fabricated
answers.

## Setup

You need a TokenPath API key — free at [platform.tokenpath.ai](https://platform.tokenpath.ai)
(10M attributed tokens on signup, no card required). Export it as `TOKENPATH_API_KEY`
before running this notebook.

In [1]:
%pip install -q requests pandas

In [2]:
import os
import re

import requests

API_URL = "https://api.tokenpath.ai"
API_KEY = os.environ.get("TOKENPATH_API_KEY")
assert API_KEY, (
    "Set TOKENPATH_API_KEY first — grab a free key (10M tokens, no card) "
    "at https://platform.tokenpath.ai"
)


def attribute(document, question, answer, spans, **options):
    """Resolve each answer character span to the source span that produced it."""
    response = requests.post(
        f"{API_URL}/v1/attributions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "document": document,
            "question": question,
            "answer": answer,
            "spans": spans,
            **options,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

### Splitting an answer into claims

TokenPath resolves the character spans *you* send. The simplest useful unit is a
sentence: split the answer on sentence boundaries and attribute each one.

In [3]:
def claim_spans(text):
    """Sentence-level [start, end) character spans over `text`."""
    boundary = re.compile(r"[.!?][\"\')\]]*(?=\s|$)|\n")
    raw, start = [], 0
    for m in boundary.finditer(text):
        raw.append((start, m.end()))
        start = m.end()
    if start < len(text):
        raw.append((start, len(text)))
    spans = []
    for s, e in raw:
        segment = text[s:e]
        if segment.strip():
            left = len(segment) - len(segment.lstrip())
            right = len(segment) - len(segment.rstrip())
            spans.append([s + left, e - right])
    return spans


demo = "Revenue grew 18%. Margin improved."
[demo[s:e] for s, e in claim_spans(demo)]

['Revenue grew 18%.', 'Margin improved.']

## The metric

Split the answer into claims, attribute all of them in one call, and aggregate
the per-claim confidences:

- **`min_confidence`** — the harsh one. One fabricated claim tanks it. Use this
  to *gate*.
- **`mean_confidence`** — graded quality. Use this to *rank* prompts, models, or
  retrieval configs.
- **`frac_grounded`** — fraction of claims at or above a threshold. Easy to read
  on a dashboard.

In [4]:
def groundedness(document, question, answer, threshold=0.8):
    """Score how grounded `answer` is in `document`. Returns scalar metrics."""
    spans = claim_spans(answer)
    result = attribute(document, question, answer, spans)
    confidences = [
        item["source"]["confidence"] if item["source"] else 0.0
        for item in result["spans"]
    ]
    return {
        "n_claims": len(confidences),
        "min_confidence": min(confidences),
        "mean_confidence": sum(confidences) / len(confidences),
        "frac_grounded": sum(c >= threshold for c in confidences) / len(confidences),
    }

## Score a batch of candidate answers

Same document, same question, four answers of decreasing honesty — the kind of
spread your eval set should contain.

In [5]:
DOCUMENT = """NORTHWIND TRADERS — Q3 2025 SHAREHOLDER NOTE

Revenue grew 18% year over year to $412 million, driven primarily by the
Enterprise segment, which expanded 31%. Gross margin improved to 64.2%,
up from 61.8% a year ago. The company closed the quarter with 2,847
employees, up from 2,610 at the end of Q2.

Operating expenses rose 9% to $198 million, reflecting continued
investment in the Fabrikam integration, which remains on track to
complete in Q1 2026. The board approved a $50 million share buyback
program. No dividend was declared this quarter.

CEO Elena Vasquez said the company expects "mid-teens revenue growth"
for the full fiscal year, citing a record $1.2 billion contracted
backlog. Guidance assumes no material FX headwinds."""

QUESTION = "Summarize Northwind's Q3 results."

In [6]:
CANDIDATES = {
    "grounded": (
        "Northwind's revenue grew 18% year over year to $412 million, led by "
        "31% growth in the Enterprise segment. Gross margin improved to 64.2%. "
        "The board approved a $50 million share buyback."
    ),
    "partly_fabricated": (
        "Northwind's revenue grew 18% year over year to $412 million. The "
        "company declared a $0.10 dividend and announced a new office in "
        "Singapore. Gross margin improved to 64.2%."
    ),
    "distorted": (
        "Northwind's revenue grew 24% year over year to $450 million. Gross "
        "margin improved to 64.2%. The board approved a $75 million share "
        "buyback."
    ),
    "fabricated": (
        "Northwind announced a merger with Contoso and plans to go private in "
        "2027. The CEO resigned effective immediately."
    ),
}

import pandas as pd

scores = pd.DataFrame(
    {name: groundedness(DOCUMENT, QUESTION, answer) for name, answer in CANDIDATES.items()}
).T.sort_values("min_confidence", ascending=False)
scores.round(3)

,n_claims,min_confidence,mean_confidence,frac_grounded
grounded,3.0,0.983,0.988,1.000
distorted,3.0,0.954,0.973,1.000
partly_fabricated,3.0,0.636,0.866,0.667
fabricated,2.0,0.055,0.312,0.000


## Reading the table — and the one thing attribution can't tell you

- **Fabrication collapses the metric.** Claims with no source to rest on lose
  their attribution mass, so `min_confidence` and `frac_grounded` drop sharply.
- **Distortion does not.** Look at the `distorted` row: "grew 24% to $450
  million" attributes *confidently* — to the sentence that says 18% and $412
  million, because that's the span it was generated from. Attribution localizes;
  it doesn't judge. To catch distortion, add one narrow verifier call per claim
  (claim vs. its mapped source span — see the
  [grounding-gate notebook](01-grounding-gate.ipynb)). The attribution step is
  what makes that verifier cheap and reliable: it grades a sentence against a
  sentence, not an essay against a document.

So: use `groundedness()` as the always-on, deterministic signal, and reserve
LLM judgment for the tiny claim-vs-span comparisons attribution hands you.

## Dropping it into your harness

The metric is a pure function of `(document, question, answer)`, so it fits any
eval framework as a custom scorer — pytest asserts, promptfoo/braintrust/LangSmith
custom metrics, or a reward signal in an RL loop.

In [7]:
def eval_case(document, question, answer, min_confidence=0.8):
    """Pytest-style check: fail if any claim in the answer is unsupported."""
    report = groundedness(document, question, answer)
    assert report["min_confidence"] >= min_confidence, (
        f"unsupported claim in answer (min_confidence="
        f"{report['min_confidence']:.2f}): {answer!r}"
    )
    return report


eval_case(DOCUMENT, QUESTION, CANDIDATES["grounded"])

{'n_claims': 3,
 'min_confidence': 0.982941,
 'mean_confidence': 0.988429,
 'frac_grounded': 1.0}

## Notes for real eval sets

- **Cost scales with tokens, not calls.** $1 per 1M attributed tokens — a
  500-case eval over 2k-token contexts runs for about a dollar.
- **Deterministic** — the same input gives the same scores, so metric deltas
  between runs are signal, not judge noise.
- **Rank retrieval, not just generation.** A low `mean_confidence` with correct
  answers often means the model answered from parametric memory because
  retrieval returned the wrong passage — attribution catches that silently
  broken state.

---

## Where to go next

- **API reference & guides** — [docs.tokenpath.ai](https://docs.tokenpath.ai)
- **Bugs / broken examples** — [open an issue](https://github.com/tokenpath/cookbook/issues)
- **"How do I…?"** — [start a discussion](https://github.com/tokenpath/cookbook/discussions)
- **Email** — support@tokenpath.ai